# M6 — Isolated gold-train linguistic generation engine

This standalone benchmark method learns only from the finalized linguistic annotations of the locked gold **training** split. It does not load M5 mappings and does not apply M1 rules. Any later `M5 + M6` or `M1 + M6` combination must be exported as a separately named ablation.

In [1]:
from pathlib import Path
from dataclasses import replace
import json
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.annotation.linguistic_annotation_core import build_occurrences
from src.config import (
    FORMAL_TOKEN_LOOKUP_PATH, FORMAL_TOKEN_LOOKUP_SEED_PATH,
    GOLD_SPLIT_MANIFEST_PATH, GOLD_TRAIN_PATH, LINGUISTIC_ANNOTATIONS_PATH,
    M6_GOLD_MAPPINGS_PATH, M6_LINGUISTIC_MANIFEST_PATH,
    M6_LINGUISTIC_PATH, M6_PROFILE_PATH, M6_REVIEW_PATH,
    M6_TRANSFORMATIONS_PATH, TRAIN_PARQUET_PATH,
)
from src.generation.benchmark_baselines import load_formal_sources
from src.generation.gold_linguistic_engine import (
    LinguisticConfig, build_m6_pairs, build_token_lookup,
    finalize_m6_manifest, learn_linguistic_profile, write_m6_outputs,
)

## Configuration

The thresholds retain observed reusable mappings and only generalize character transformations seen at least twice with at least 60% dominance for the same POS-conditioned trigger. `max_edits_per_sentence=None` removes the artificial cap but still samples the empirical number of changed occurrences per gold-training sentence; it does not blindly apply every possible edit. Export and finalization are deliberately disabled by default.

In [2]:
CONFIG = LinguisticConfig(
    min_mapping_count=1,
    min_mapping_dominance=0.50,
    min_transformation_count=2,
    min_transformation_dominance=0.60,
    max_edits_per_sentence=None,  # Diagnostic uncapped profile; observed maximum is 15.
    seed=2026,
)
PREVIEW_ROWS = 500
EXPORT_M6 = True
OVERWRITE_M6 = False
FINALIZE_M6_AFTER_REVIEW = True
CONFIG

LinguisticConfig(min_mapping_count=1, min_mapping_dominance=0.5, min_transformation_count=2, min_transformation_dominance=0.6, max_edits_per_sentence=None, seed=2026, allowed_target_language='wo', allowed_source_language='wo', generalizable_error_types=('diacritic', 'deletion', 'insertion', 'substitution', 'repetition'))

## Load and audit the finalized train-only evidence

This cell fails if an annotation is missing, orphaned, duplicated, mismatched with gold train, or sourced outside the locked training IDs. Development and test files are never loaded as learning evidence.

In [3]:
gold_train = pd.read_csv(GOLD_TRAIN_PATH, dtype={'source_index': 'string'}, keep_default_na=False)
annotations = pd.read_csv(
    LINGUISTIC_ANNOTATIONS_PATH, dtype={'source_index': 'string'}, keep_default_na=False
)
latest_annotations = annotations.drop_duplicates('occurrence_id', keep='last')
occurrences = build_occurrences(gold_train)
assert not annotations.occurrence_id.duplicated().any(), 'Duplicate linguistic occurrence IDs'
assert set(latest_annotations.occurrence_id) == set(occurrences.occurrence_id), (
    'Linguistic annotation coverage no longer matches gold train'
)
comparison = latest_annotations.merge(
    occurrences[['occurrence_id', 'informal_token', 'formal_token']],
    on='occurrence_id', suffixes=('_annotation', '_gold'), validate='one_to_one',
)
assert comparison.informal_token_annotation.eq(comparison.informal_token_gold).all()
assert comparison.formal_token_annotation.eq(comparison.formal_token_gold).all()
profile = learn_linguistic_profile(
    latest_annotations, gold_train.source_index, config=CONFIG
)
print(f'Gold train sentences: {len(gold_train):,}')
print(f'Finalized linguistic occurrences: {len(latest_annotations):,}')
print(json.dumps(profile.training_summary, indent=2, ensure_ascii=False))
assert profile.training_summary['uses_m5_outputs'] is False
assert profile.training_summary['uses_m1_rules'] is False

Gold train sentences: 142
Finalized linguistic occurrences: 1,315
{
  "annotation_rows": 1315,
  "eligible_occurrences": 865,
  "eligible_changed_occurrences": 588,
  "gold_train_source_count": 142,
  "mapping_count": 171,
  "transformation_count": 105,
  "uses_m5_outputs": false,
  "uses_m1_rules": false
}


In [4]:
print('Empirical edits per gold-train sentence (artificial cap disabled):')
display(pd.Series(profile.edit_budgets).value_counts().sort_index().rename('sentence_count'))
print('Learned lexical/phrase mappings:')
display(profile.mappings.head(25))
print('Learned POS-conditioned character transformations:')
display(profile.transformations.head(25))
display(pd.DataFrame(profile.pos_error_profile).T)

Empirical edits per gold-train sentence (artificial cap disabled):


1     15
2     32
3     29
4     15
5     21
6      8
7      9
8      4
10     1
12     4
13     1
14     2
15     1
Name: sentence_count, dtype: int64

Learned lexical/phrase mappings:


,formal_token,informal_token,count,formal_total,dominance,confidence,mapping_source,pos_candidates_json,error_types_json
0,am na,amna,4,4,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""phonetic_spelling"", ""spacing_merge""]"
1,na ma,nama,3,3,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""phonetic_spelling"", ""spacing_merge""]"
2,soof na,sofna,3,4,0.750000,0.750000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""deletion"", ""phonetic_spelling"", ""spacing_mer..."
3,da ñoo,dagno,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""deletion"", ""diacritic"", ""spacing_merge"", ""su..."
4,di naa ko,dinako,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""deletion"", ""spacing_merge""]"
5,maa,ma,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""deletion""]"
6,moo ngi,mongui,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""deletion"", ""insertion"", ""phonetic_spelling"",..."
7,naa ko,nako,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""deletion"", ""spacing_merge""]"
8,nak,nk,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""abbreviation""]"
9,nee na,neina,2,2,1.000000,1.000000,gold_train_linguistic_mapping,"[""UNKNOWN""]","[""phonetic_spelling"", ""spacing_merge"", ""substi..."


Learned POS-conditioned character transformations:


,pos,error_type,operation,formal_text,informal_text,left_context,right_context,position_kind,count,trigger_total,dominance,confidence,error_weight,mapping_source
0,UNKNOWN,deletion,delete,a,,a,$,suffix,12,12,1.000000,1.000000,0.176119,gold_train_linguistic_transformation
1,DET,insertion,insert,,o,k,u,internal,8,8,1.000000,1.000000,0.478261,gold_train_linguistic_transformation
2,PRON,deletion,delete,o,,o,m,internal,7,7,1.000000,1.000000,0.178571,gold_train_linguistic_transformation
3,VERB,insertion,insert,,o,m,u,internal,7,7,1.000000,1.000000,0.161731,gold_train_linguistic_transformation
4,VERB,substitution,replace,x,kh,a,$,suffix,7,9,0.777778,0.777778,0.266515,gold_train_linguistic_transformation
5,NOUN,diacritic,replace,jà,dia,^,m,prefix,6,6,1.000000,1.000000,0.166667,gold_train_linguistic_transformation
6,NOUN,insertion,replace,ja,di,i,a,internal,6,6,1.000000,1.000000,0.196296,gold_train_linguistic_transformation
7,VERB,diacritic,replace,ë,eu,b,g,internal,6,6,1.000000,1.000000,0.150342,gold_train_linguistic_transformation
8,VERB,substitution,replace,ex,kh,e,$,suffix,6,6,1.000000,1.000000,0.266515,gold_train_linguistic_transformation
9,INTJ,deletion,delete,o,,o,$,suffix,5,5,1.000000,1.000000,0.280702,gold_train_linguistic_transformation


,eligible_occurrences,changed_occurrences,change_rate,error_counts,error_probabilities
ADJ,1,1,1.0,"{'diacritic': 1, 'phonetic_spelling': 1, 'subs...","{'diacritic': 0.3333333333333333, 'phonetic_sp..."
ADP,12,6,0.5,"{'insertion': 2, 'phonetic_spelling': 1, 'repe...","{'insertion': 0.25, 'phonetic_spelling': 0.125..."
ADV,21,15,0.714286,"{'abbreviation': 1, 'deletion': 2, 'diacritic'...","{'abbreviation': 0.029411764705882353, 'deleti..."
CCONJ,37,19,0.513514,"{'abbreviation': 2, 'deletion': 6, 'diacritic'...","{'abbreviation': 0.06451612903225806, 'deletio..."
DET,94,26,0.276596,"{'deletion': 3, 'insertion': 22, 'phonetic_spe...","{'deletion': 0.06521739130434782, 'insertion':..."
INTJ,36,26,0.722222,"{'abbreviation': 1, 'deletion': 16, 'diacritic...","{'abbreviation': 0.017543859649122806, 'deleti..."
NOUN,122,106,0.868852,"{'deletion': 30, 'diacritic': 45, 'insertion':...","{'deletion': 0.1111111111111111, 'diacritic': ..."
NUM,1,1,1.0,"{'diacritic': 1, 'substitution': 1}","{'diacritic': 0.5, 'substitution': 0.5}"
PART,4,2,0.5,"{'insertion': 2, 'phonetic_spelling': 2}","{'insertion': 0.5, 'phonetic_spelling': 0.5}"
PRON,60,39,0.65,"{'deletion': 15, 'diacritic': 11, 'insertion':...","{'deletion': 0.17857142857142858, 'diacritic':..."


## Build a deterministic preview

The curated lookup takes precedence over the multilingual seed. French, named entities, ambiguous entries, acronyms, numbers, and other protected tokens are not modified. Unknown tokens may receive an exact reviewed lexical or phrase mapping but cannot receive a generalized POS template. Spacing-merge units are intentionally treated as multiword phrase mappings, so their `UNKNOWN` POS does not prevent them from being learned or applied.

In [5]:
curated_lookup = pd.read_csv(FORMAL_TOKEN_LOOKUP_PATH, keep_default_na=False)
seed_lookup = pd.read_csv(FORMAL_TOKEN_LOOKUP_SEED_PATH, keep_default_na=False)
token_lookup = build_token_lookup(curated_lookup, seed_lookup)
formal_sources = load_formal_sources(TRAIN_PARQUET_PATH)
preview = build_m6_pairs(
    formal_sources.head(PREVIEW_ROWS), profile, token_lookup, config=CONFIG
)
capped_config = replace(CONFIG, max_edits_per_sentence=3)
capped_profile = learn_linguistic_profile(
    latest_annotations, gold_train.source_index, config=capped_config
)
capped_preview = build_m6_pairs(
    formal_sources.head(PREVIEW_ROWS), capped_profile, token_lookup, config=capped_config
)
comparison_table = pd.DataFrame({
    'profile': ['capped_at_3', 'uncapped_empirical'],
    'changed_rows': [int(capped_preview.changed.sum()), int(preview.changed.sum())],
    'changed_rate': [float(capped_preview.changed.mean()), float(preview.changed.mean())],
    'total_edits': [int(capped_preview.edit_count.sum()), int(preview.edit_count.sum())],
    'maximum_row_edits': [int(capped_preview.edit_count.max()), int(preview.edit_count.max())],
})
display(comparison_table)
print(f'Preview rows: {len(preview):,}')
print(f'Changed: {preview.changed.sum():,} ({preview.changed.mean():.1%})')
print(f'Lexical/phrase edits: {preview.lexical_mapping_edit_count.sum():,}')
print(f'Generalized transformation edits: {preview.transformation_edit_count.sum():,}')
display(preview.loc[preview.changed, [
    'formal_wolof', 'informal_wolof', 'error_types', 'applied_transformations'
]].head(30))

,profile,changed_rows,changed_rate,total_edits,maximum_row_edits
0,capped_at_3,414,0.828,898,3
1,uncapped_empirical,414,0.828,1065,9


Preview rows: 500
Changed: 414 (82.8%)
Lexical/phrase edits: 428
Generalized transformation edits: 637


,formal_wolof,informal_wolof,error_types,applied_transformations
0,Bataaxal bii jëwriñu lu ajju ci mbiru bitim ré...,Bataaxal bii jëwriñu lou ajju ci mbiru bitim r...,"[""deletion"", ""insertion"", ""phonetic_spelling""]","[{""char_end"": 23, ""char_start"": 21, ""confidenc..."
1,"""Amuñu woon benn jot ngir rawal sunu bopp.","""Amuñu woonne be jot ngir rawal sunu bopp.","[""diacritic"", ""repetition""]","[{""char_end"": 11, ""char_start"": 7, ""confidence..."
2,"Ekost sa dëkk la te bëggna nu nga toog fi.""","Ekost sa dëkk la te bëggna nu ngua toog fi.""","[""deletion""]","[{""char_end"": 33, ""char_start"": 30, ""confidenc..."
4,ñaata at nga am,ñaata at ga ame,"[""deletion"", ""insertion""]","[{""char_end"": 12, ""char_start"": 9, ""confidence..."
5,"Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan Nja...","Abu Usmaan Si, Mamadu Yoro Jàllo ak Usmaan Nja...","[""diacritic"", ""insertion"", ""phonetic_spelling""...","[{""char_end"": 52, ""char_start"": 49, ""confidenc..."
6,Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm...,Ni ñu ko tàmm a waxe fii : boo xamatul foo jëm...,"[""deletion"", ""diacritic"", ""phonetic_spelling"",...","[{""char_end"": 66, ""char_start"": 62, ""confidenc..."
7,"Zambie réew la mu bokk ci ONU, Union Africaine...","Zambie réew la mou bokk ci ONU, Union Africain...","[""insertion"", ""phonetic_spelling""]","[{""char_end"": 17, ""char_start"": 15, ""confidenc..."
10,"Beneen wuute bi ci am mooy, bu jigéen ak néew ...","Beneen wuute bi ci ame mooy, bu jigéen ak néew...","[""insertion""]","[{""char_end"": 21, ""char_start"": 19, ""confidenc..."
11,Góor gu mag gi ak géej gi,Góor gu mag gi ak guej gi,"[""diacritic""]","[{""char_end"": 22, ""char_start"": 18, ""confidenc..."
13,"Ki mën ci jaww ji ak bu wanee ginnaaw kãa yi, ...","Ki mën ci djaww ji ak bu wanee ginnaaw kãa yi,...","[""diacritic"", ""insertion""]","[{""char_end"": 51, ""char_start"": 46, ""confidenc..."


## Export the standalone M6 condition

Inspect the capped-versus-uncapped comparison and the uncapped examples first. `None` preserves empirical gold-train counts up to 15 edits, so it is useful diagnostically but may over-noisify formal sentences. Choose the final cap in `CONFIG`, rerun the profile and preview cells, then set `EXPORT_M6=True`. The manifest records the chosen value and explicitly records that M5 and M1 were not used. Existing artifacts are protected unless `OVERWRITE_M6=True`.

In [6]:
if EXPORT_M6:
    output_paths = (
        M6_LINGUISTIC_PATH, M6_LINGUISTIC_MANIFEST_PATH, M6_PROFILE_PATH,
        M6_GOLD_MAPPINGS_PATH, M6_TRANSFORMATIONS_PATH, M6_REVIEW_PATH,
    )
    existing = [path for path in output_paths if path.exists()]
    if existing and not OVERWRITE_M6:
        raise FileExistsError(f'Refusing to overwrite M6 artifacts: {existing}')
    m6_pairs = build_m6_pairs(formal_sources, profile, token_lookup, config=CONFIG)
    assert len(m6_pairs) == len(formal_sources)
    assert m6_pairs.source_id.tolist() == formal_sources.source_id.tolist()
    manifest = write_m6_outputs(
        m6_pairs, profile, output_path=M6_LINGUISTIC_PATH,
        mappings_path=M6_GOLD_MAPPINGS_PATH,
        transformations_path=M6_TRANSFORMATIONS_PATH,
        profile_path=M6_PROFILE_PATH, review_path=M6_REVIEW_PATH,
        manifest_path=M6_LINGUISTIC_MANIFEST_PATH, config=CONFIG,
        input_paths={
            'formal_sources': TRAIN_PARQUET_PATH,
            'gold_train': GOLD_TRAIN_PATH,
            'gold_split_manifest': GOLD_SPLIT_MANIFEST_PATH,
            'linguistic_annotations': LINGUISTIC_ANNOTATIONS_PATH,
            'curated_lookup': FORMAL_TOKEN_LOOKUP_PATH,
            'multilingual_lookup_seed': FORMAL_TOKEN_LOOKUP_SEED_PATH,
        },
    )
    assert manifest['uses_m5_outputs'] is False
    assert manifest['uses_m1_rules'] is False
    display(manifest)
else:
    print('M6 export disabled. Inspect the preview, then set EXPORT_M6=True.')

{'method': 'm6_gold_linguistic_engine',
 'created_at_utc': '2026-08-29T21:12:46.323787+00:00',
 'provisional': True,
 'learning_scope': 'finalized locked gold train linguistic annotations only',
 'uses_m5_outputs': False,
 'uses_m1_rules': False,
 'leakage_status': 'no development/test annotations or M5 outputs consumed',
 'rows': 17777,
 'changed_rows': 14215,
 'unchanged_rows': 3562,
 'lexical_mapping_edits': 14809,
 'transformation_edits': 22119,
 'configuration': {'min_mapping_count': 1,
  'min_mapping_dominance': 0.5,
  'min_transformation_count': 2,
  'min_transformation_dominance': 0.6,
  'max_edits_per_sentence': None,
  'seed': 2026,
  'allowed_target_language': 'wo',
  'allowed_source_language': 'wo',
  'generalizable_error_types': ('diacritic',
   'deletion',
   'insertion',
   'substitution',
   'repetition')},
 'training_summary': {'annotation_rows': 1315,
  'eligible_occurrences': 865,
  'eligible_changed_occurrences': 588,
  'gold_train_source_count': 142,
  'mapping_cou

## Manual quality gate

After export, manually complete every row of `data/synthetic/m6_gold_linguistic_review_sample.csv`: `meaning_preserved`, `informal_plausibility_1_5`, `severity_1_5`, `synthetic_artifact`, and optional `review_notes`. Do not inspect gold development or test targets while rating. When all required cells are complete, set `FINALIZE_M6_AFTER_REVIEW=True`.

In [7]:
if FINALIZE_M6_AFTER_REVIEW:
    finalized_manifest = finalize_m6_manifest(
        M6_LINGUISTIC_MANIFEST_PATH, M6_REVIEW_PATH
    )
    display(finalized_manifest)
else:
    print('M6 finalization disabled until the 100-row review is complete.')

ValueError: M6 review is incomplete: informal_plausibility_1_5